# 06 PhoBERT Title Embedding + PCA Compression

Notebook nay chay tren Google Colab/GPU de tao embedding cho title/headline tin tuc. Raw PhoBERT embedding 768 chieu se duoc nen xuong 64 chieu bang PCA truoc khi dua vao GNN.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import shutil
import subprocess

REPO_URL = 'https://github.com/vuhongthuy59-cell/colabvscode_kltn.git'
PROJECT_DIR = Path('/content/colabvscode_kltn')
DRIVE_ROOT = Path('/content/drive/MyDrive/KLTN_GNN')

if PROJECT_DIR.exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)

%cd /content/colabvscode_kltn

In [ ]:
!pip -q install transformers sentencepiece scikit-learn pandas tqdm

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

NEWS_FILE = PROJECT_DIR / 'outputs/local/02_news_data/news_articles.csv'
RAW_OUTPUT = DRIVE_ROOT / 'outputs/local/02_news_data/news_title_phobert_768.npy'
ID_OUTPUT = DRIVE_ROOT / 'outputs/local/02_news_data/news_title_phobert_article_ids.csv'
PCA_OUTPUT = DRIVE_ROOT / 'outputs/local/02_news_data/news_title_embedding_pca.csv'
LOCAL_PCA_OUTPUT = PROJECT_DIR / 'outputs/local/02_news_data/news_title_embedding_pca.csv'

PCA_DIM = 64
BATCH_SIZE = 64
MAX_LENGTH = 96

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', device)
articles = pd.read_csv(NEWS_FILE)
articles['article_id'] = articles['article_id'].astype(str)
text_col = 'title_clean' if 'title_clean' in articles.columns else 'title'
texts = articles[text_col].fillna('').astype(str).tolist()
article_ids = articles['article_id'].tolist()
print('articles =', len(articles), 'text_col =', text_col)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('vinai/phobert-base', use_fast=False)
model = AutoModel.from_pretrained('vinai/phobert-base').to(device)
model.eval()

embeddings = []
with torch.no_grad():
    for start in tqdm(range(0, len(texts), BATCH_SIZE)):
        batch_texts = texts[start:start + BATCH_SIZE]
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors='pt',
        )
        encoded = {key: value.to(device) for key, value in encoded.items()}
        output = model(**encoded)
        cls_embedding = output.last_hidden_state[:, 0, :].detach().cpu().numpy().astype(np.float32)
        embeddings.append(cls_embedding)

embedding_matrix = np.vstack(embeddings).astype(np.float32)
RAW_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
np.save(RAW_OUTPUT, embedding_matrix)
pd.DataFrame({'article_id': article_ids}).to_csv(ID_OUTPUT, index=False)
print('raw shape =', embedding_matrix.shape)
print('saved raw =', RAW_OUTPUT)

In [ ]:
scaled = StandardScaler().fit_transform(embedding_matrix)
compressed = PCA(n_components=PCA_DIM, random_state=42).fit_transform(scaled).astype(np.float32)

pca_df = pd.DataFrame({'article_id': article_ids})
for idx in range(PCA_DIM):
    pca_df[f'phobert_pca_{idx + 1:03d}'] = compressed[:, idx]

PCA_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
pca_df.to_csv(PCA_OUTPUT, index=False)
LOCAL_PCA_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
pca_df.to_csv(LOCAL_PCA_OUTPUT, index=False)
print('compressed shape =', compressed.shape)
print('saved Drive PCA =', PCA_OUTPUT)
print('saved local repo PCA =', LOCAL_PCA_OUTPUT)

## Buoc tiep theo

Sau khi notebook nay chay xong, copy file sau ve VSCode/local project neu can rebuild graph o local:

`MyDrive/KLTN_GNN/outputs/local/02_news_data/news_title_embedding_pca.csv`

Sau do chay lai:

`python scripts/05_build_event_graph_dataset.py`

roi upload `outputs/local/05_event_graph_dataset/graph_snapshots.pt` moi len Drive truoc khi train GNN.